Attention = softmax(QK^T/sqrt(d_k))V

In [2]:
import torch
import torch.nn as nn

# 6 tokens, each 3d 
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

attn_scores = inputs @ inputs.T     # 6*3 matmul 3*6 == [6*6]
print(attn_scores.shape, "\n", attn_scores)
attn_weights = torch.softmax(attn_scores, dim = -1)     #6*6 normalized scores
print(attn_weights.shape, "\n", attn_weights)

all_context_vecs = attn_weights @ inputs    # 6*6 matmul 6*3 == 6*3
print(all_context_vecs.shape, "\n", all_context_vecs)

torch.Size([6, 6]) 
 tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
torch.Size([6, 6]) 
 tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
torch.Size([6, 3]) 
 tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [ ]:
# suppose each token x is 3d, so in_dim is 3, out_dim is 2
class NaiveAttention(nn.Module):
    def __init__(self, in_dim:int, out_dim: int):
        super().__init__()

        self.W_query = nn.Parameter(torch.rand(in_dim, out_dim))
        self.W_key = nn.Parameter(torch.rand(in_dim, out_dim))
        self.W_value = nn.Parameter(torch.rand(in_dim, out_dim))

    def forward(self, x):
        queries = x @ self.W_query          # 1*3 matmul 3*2 == 1*2
        keys = x @ self.W_key
        values = x @ self.W_value

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values

        return context_vec

torch.manual_seed(123)
selfAttn = NaiveAttention(3, 2)
print(selfAttn(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [17]:
# suppose each token x is 3d, so in_dim is 3, out_dim is 2
class NaiveAttentionV2(nn.Module):
    def __init__(self, in_dim:int, out_dim: int, qkv_bias=False):
        super().__init__()

        self.W_query = nn.Linear(in_dim, out_dim, bias=qkv_bias)
        self.W_key = nn.Linear(in_dim, out_dim, bias=qkv_bias)
        self.W_value = nn.Linear(in_dim, out_dim, bias=qkv_bias)

    def forward(self, x):
        print("Type: ", x.type, " and ", self.W_value.type)
        queries = self.W_query(x)          # 1*3 matmul 3*2 == 1*2
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values

        return context_vec

torch.manual_seed(123)
selfAttn = NaiveAttentionV2(3, 2)
print(selfAttn(inputs))

Type:  <built-in method type of Tensor object at 0x1386658f0>  and  <bound method Module.type of Linear(in_features=3, out_features=2, bias=False)>
tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)


In [21]:
hudai = torch.rand(5, 5)
mask = torch.triu(torch.ones(5, 5), diagonal=1)
print(mask)
hudai = hudai.masked_fill(mask.bool(), -torch.inf)
print(hudai)

tensor([[0., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1.],
        [0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0.]])
tensor([[0.8459,   -inf,   -inf,   -inf,   -inf],
        [0.9010, 0.3950,   -inf,   -inf,   -inf],
        [0.2185, 0.3834, 0.3720,   -inf,   -inf],
        [0.7475, 0.4979, 0.8549, 0.2438,   -inf],
        [0.4536, 0.4130, 0.5585, 0.1170, 0.5578]])


Causal self attention with dropout

In [30]:
class CausalAttention(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, context_length: int, drop_rate:0.2, qkv_bias=False):
        super().__init__()
        self.d_out = out_dim
        self.W_query = nn.Linear(in_dim, out_dim, qkv_bias)
        self.W_key = nn.Linear(in_dim, out_dim, qkv_bias)
        self.W_value = nn.Linear(in_dim, out_dim, qkv_bias)
        self.dropout = nn.Dropout(drop_rate)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # print(queries.shape, " ", keys.shape, " key transpose: ", keys.transpose(2, 1))
        attn_scores = queries @ keys.transpose(2, 1)
        attn_scores.masked_fill(self.mask.bool() [:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        context_vec = attn_weights @ values

        return context_vec

torch.manual_seed(123)
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

d_in = batch.shape[2]
d_out = 2
context_length = batch.shape[1]

cattn = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = cattn(batch)
print("context_vecs shape: ", context_vecs.shape, "\n", context_vecs)

torch.Size([2, 6, 3])
context_vecs shape:  torch.Size([2, 6, 2]) 
 tensor([[[-0.5337, -0.1051],
         [-0.5323, -0.1080],
         [-0.5323, -0.1079],
         [-0.5297, -0.1076],
         [-0.5311, -0.1066],
         [-0.5299, -0.1081]],

        [[-0.5337, -0.1051],
         [-0.5323, -0.1080],
         [-0.5323, -0.1079],
         [-0.5297, -0.1076],
         [-0.5311, -0.1066],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)


In [39]:
class MultiHeadAttention(nn.Module):
    def __init__(self, in_dim, out_dim, context_length, drop_rate, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(in_dim, out_dim, context_length, drop_rate, qkv_bias) for _ in range(num_heads)]
        )
    def forward(self, x):
        head_shapes = [head(x).shape for head in self.heads]
        print(head_shapes)
        return torch.cat([head(x) for head in self.heads], dim=-1)
        
torch.manual_seed(123)
context_length = batch.shape[1]
d_in, d_out = 3, 2

mha = MultiHeadAttention(d_in, d_out, context_length,  0.0, num_heads = 2)
context_vecs = mha(batch)

print("context_vecs shape: ", context_vecs.shape, "\n", context_vecs)

[torch.Size([2, 6, 2]), torch.Size([2, 6, 2])]
context_vecs shape:  torch.Size([2, 6, 4]) 
 tensor([[[-0.5337, -0.1051,  0.5085,  0.3508],
         [-0.5323, -0.1080,  0.5084,  0.3508],
         [-0.5323, -0.1079,  0.5084,  0.3506],
         [-0.5297, -0.1076,  0.5074,  0.3471],
         [-0.5311, -0.1066,  0.5076,  0.3446],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.5337, -0.1051,  0.5085,  0.3508],
         [-0.5323, -0.1080,  0.5084,  0.3508],
         [-0.5323, -0.1079,  0.5084,  0.3506],
         [-0.5297, -0.1076,  0.5074,  0.3471],
         [-0.5311, -0.1066,  0.5076,  0.3446],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(123)
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)
batch = torch.stack((inputs, inputs), dim=0)
d_in = batch.shape[-1]
d_out = 2
context_length = batch.shape[1]

class MultiHeadAttentionV2(nn.Module):
    def __init__(self, d_in, d_out, context_length, drop_rate, num_heads, qkv_bias=False):
        super().__init__()
        assert(d_out % num_heads == 0), "out dimension must be divisible by num of heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        
        self.W_query = nn.Linear(d_in, d_out, qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.drop = nn.Dropout(drop_rate)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # [b, num_tokens, d_out]    [2, 6, 2]
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)    # [b, num_tokens, num_head, head_dim]   [2, 6, 2, 1]
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        queries = queries.transpose(1, 2)   # [b, num_head, num_tokens, head_dim]   [2, 2, 6, 1]
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)    # [2, 2, 6, 1] @ [2, 2, 1, 6] == [2, 2, 6, 6]
        # print("attn_scores shape: ", attn_scores.shape)

        mask_bool = self.mask.bool() [:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)     # [2, 2, 6, 6]
        attn_weights = self.drop(attn_weights)
        # print("attn_weights shape: ", attn_weights.shape)

        context_vecs = (attn_weights @ values)      # [2, 2, 6, 6] @ [2, 2, 6, 1] == [2, 2, 6, 1]
        # print("context_vecs shape: ", context_vecs.shape)
        context_vecs = context_vecs.transpose(1, 2)     # [2, 6, 2, 1]
        # print("context_vecs shape: ", context_vecs.shape)

        context_vecs = context_vecs.contiguous().view(b, num_tokens, self.d_out)
        context_vecs = self.out_proj(context_vecs)

        return context_vecs
    
mha2 = MultiHeadAttentionV2(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha2(batch)

# print(batch)
print("context_vecs shape: ", context_vecs.shape, "\n", context_vecs)

context_vecs shape:  torch.Size([2, 6, 2]) 
 tensor([[[0.2595, 0.4014],
         [0.2583, 0.4014],
         [0.2583, 0.4014],
         [0.2575, 0.4031],
         [0.2582, 0.4026],
         [0.2575, 0.4028]],

        [[0.2595, 0.4014],
         [0.2583, 0.4014],
         [0.2583, 0.4014],
         [0.2575, 0.4031],
         [0.2582, 0.4026],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
